# Lab R.2 &mdash; Rerankers: search wide, then read closely

**About 25 minutes** &middot; Day 2 &middot; RAG, vector stores &amp; agent memory

Hybrid search still misses some questions at 3. Here you keep its **top 20** and add a second stage, a **reranker**, that reads the question with the candidates and picks the best 3. You try a small cross-encoder and the sandbox model, on the same 14 questions.

Run the cells in order, with **Shift + Enter**. Under each cell, **You should see** says what to
expect. The shared helpers are in `rag_kit.py`, next to this notebook.

**The result:** one table with hits, time and tokens for each method, so you can decide whether a reranker is worth its cost.

## Step 1 &mdash; The baseline

This cell rebuilds the hybrid search from Lab R.1. `measure()` runs any ranking over the 14
questions and counts hits at 1 and at 3, and the time taken.

In [ ]:
import os, warnings
warnings.filterwarnings("ignore")                    # the model libraries print a lot on first import
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import rag_kit as kit

import re, time
chunks = kit.all_chunks()
text = {c["id"]: c["text"] for c in chunks}
col = kit.build_collection(chunks)
bm25 = kit.BM25(chunks)
questions = kit.load_questions()

def rrf(ranked_lists, c=60):                         # from Lab R.1
    scores = {}
    for ranked in ranked_lists:
        for rank, chunk_id in enumerate(ranked, start=1):
            scores[chunk_id] = scores.get(chunk_id, 0) + 1 / (c + rank)
    return sorted(scores, key=lambda cid: -scores[cid])

def candidates(question, n=20):                      # hybrid search: the top 20
    return rrf([kit.vector_search(col, question, k=20), bm25.search(question, k=20)])[:n]

def measure(name, rank_fn):
    """rank_fn(question, candidate_ids) returns the same ids in a new order, best first."""
    t0, at1, at3 = time.time(), 0, 0
    for q in questions:
        ranked = rank_fn(q["question"], candidates(q["question"]))
        at1 += ranked[:1] == [q["chunk"]]
        at3 += q["chunk"] in ranked[:3]
    row = {"method": name, "hit@1": at1, "hit@3": at3, "seconds": round(time.time() - t0, 1)}
    print(row)
    return row

baseline = measure("hybrid only", lambda question, ids: ids)

**You should see:** one row with the hybrid hits at 1 and at 3. These are the numbers to beat.

## Step 2 &mdash; A cross-encoder reranker

Hybrid search made each chunk's vector before anyone asked the question. A **cross-encoder** reads the
question and one chunk **together**, and returns a relevance score. This small model is already in
the sandbox.

In [ ]:
os.environ["HF_HUB_OFFLINE"] = "1"                 # the model is already in the sandbox: do not check online
from sentence_transformers import CrossEncoder
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def cross_encoder_rerank(question, ids):
    scores = cross_encoder.predict([(question, text[cid]) for cid in ids])
    return [cid for _, cid in sorted(zip(scores, ids), key=lambda pair: -pair[0])]

ce_row = measure("cross-encoder", cross_encoder_rerank)

**You should see:** a row close to the baseline, not much better. The model learned from **web
search** questions, and a runbook reads differently. A reranker helps only when you measure that it
helps, on **your** questions.

## Step 3 &mdash; The LLM as a reranker

The sandbox model understands the question far better. Send it all 20 candidates in **one** prompt,
numbered, and ask for the numbers of the best 3. That is one model call per question, so the cell
also counts the tokens.

In [ ]:
llm_tokens = 0

def llm_rerank(question, ids):
    """Send the 20 chunks in one prompt, numbered, and ask for the best 3."""
    global llm_tokens
    numbered = "\n".join(f"[{n}] {text[cid]}" for n, cid in enumerate(ids))
    reply, tokens = kit.chat(f"Question: {question}\n\nPassages:\n{numbered}\n\n"
                             "Which 3 passages best answer the question? "
                             "Reply with their numbers only, best first, like: 4, 0, 7", max_tokens=20)
    llm_tokens += tokens
    picked = []
    for n in map(int, re.findall(r"\d+", reply)):
        if n < len(ids) and ids[n] not in picked:     # ignore numbers that are not a position
            picked.append(ids[n])
    return picked + [cid for cid in ids if cid not in picked]

q0 = questions[5]["question"]
print(q0)
print("hybrid  :", candidates(q0)[:3])
print("reranked:", llm_rerank(q0, candidates(q0))[:3])

**You should see:** one question, with the hybrid top 3 and the reranked top 3. The reranked list should start with the chunk that answers it.

## The result &mdash; is it worth it?

Run the LLM reranker on all 14 questions, and put the three rows side by side.

In [ ]:
llm_tokens = 0
llm_row = measure("LLM rerank", llm_rerank)
llm_row["tokens"] = llm_tokens

print(f"\n{'method':15}{'hit@1':>7}{'hit@3':>7}{'seconds':>9}{'tokens':>8}")
for r in (baseline, ce_row, llm_row):
    print(f"{r['method']:15}{r['hit@1']:>7}{r['hit@3']:>7}{r['seconds']:>9}{r.get('tokens', 0):>8}")
print(f"\nabout {llm_tokens // len(questions)} tokens per question")

**You should see:** the LLM reranker gives the most hits, for about 1,000 tokens per question. The
cross-encoder costs no tokens, but barely helps on these runbooks, and on the sandbox's CPU it is
no faster than the LLM.

Answer in your notes:

1. If the reranker lets you send **3** chunks to the answer step instead of **10**, how many tokens
   does that save per question? Does the reranker pay for itself?
2. An agent searches 5 times in one run. Would you rerank every search? What would you try instead?